# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Setup
Installs dependencies, defines helpers and (for Notebooks 3-7) loads the shared `config.json` written by Notebook 2. Run this cell first.

In [2]:
# ---- Environment setup (Google Colab / local Jupyter) ----
import os, sys, glob, zipfile, json, random, warnings
warnings.filterwarnings("ignore")

USE_DRIVE = False   # set True on Colab to keep artifacts (chroma_db, adapters, csv) across runtime resets
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive/")
    WORKDIR = "/content/drive/MyDrive/RAG/Dataset/Output/intent_lora"
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

%pip install -q transformers accelerate bitsandbytes peft sentence-transformers chromadb langchain langchain-community langchain-huggingface langchain-chroma rouge-score nltk scikit-learn matplotlib seaborn reportlab

def find_sop_dir():
    """Locate the folder holding the Markdown SOP files (corporate_policies/ or Dataset/sop_documents/).
    If it is not present, try to unzip a Dataset*.zip, and finally ask for an upload on Colab."""
    def _search():
        for c in ["corporate_policies", os.path.join("Dataset", "sop_documents"), "sop_documents"]:
            if os.path.isdir(c) and glob.glob(os.path.join(c, "*.md")):
                return c
        for pat in ["**/sop_documents", "**/corporate_policies"]:
            for h in glob.glob(pat, recursive=True):
                if glob.glob(os.path.join(h, "*.md")):
                    return h
        return None
    found = _search()
    if found: return found
    for z in glob.glob("*.zip") + glob.glob("**/Dataset*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    try:
        from google.colab import files
        print("SOP folder not found - please upload Dataset.zip (contains sop_documents/*.md)")
        for name in files.upload():
            if name.endswith(".zip"):
                with zipfile.ZipFile(name) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    except ImportError:
        pass
    raise FileNotFoundError("Could not find the SOP .md files. Place them in ./corporate_policies/ or ./Dataset/sop_documents/")

# ---- Load the single-source-of-truth configuration written by Notebook 2 ----
if not os.path.exists("/content/drive/MyDrive/RAG/Dataset/Output/intent_lora/config.json"):
    raise FileNotFoundError("config.json not found - run Notebook 2 (Data Preparation) first.")
cfg = json.load(open("config.json"))
MODEL_ID       = cfg["MODEL_ID"]
EMBED_MODEL    = cfg["EMBED_MODEL"]
SEED           = cfg["SEED"]
BASELINE_SYSTEM = cfg["BASELINE_SYSTEM"]
RAG_SYSTEM     = cfg["RAG_SYSTEM"]          # contains the {context} placeholder
ROUTER_SYSTEM  = cfg["ROUTER_SYSTEM"]
MAX_NEW_TOKENS = cfg["MAX_NEW_TOKENS"]
CHROMA_DIR     = cfg["CHROMA_DIR"]
COLLECTION     = cfg["COLLECTION"]
random.seed(SEED)
print("MODEL_ID:", MODEL_ID, "| EMBED_MODEL:", EMBED_MODEL, "| SEED:", SEED)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HAS_CUDA = torch.cuda.is_available()
COMPUTE_DTYPE = torch.bfloat16 if (HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print("CUDA available:", HAS_CUDA, "| compute dtype:", COMPUTE_DTYPE)
if not HAS_CUDA:
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU. Falling back to (slow) CPU float32.")

def load_base_model(model_id=None, quantize=True):
    """Load tokenizer + base model. 4-bit NF4 quantisation on GPU (about 1 GB VRAM for the 1.5B model)."""
    model_id = model_id or MODEL_ID
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"                       # required for batched decoder-only generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if HAS_CUDA and quantize:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
        model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map={"": 0})
    elif HAS_CUDA:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=COMPUTE_DTYPE, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
    model.eval()
    return tok, model

@torch.no_grad()
def generate_chat(model, tok, messages_list, max_new_tokens=None, batch_size=16):
    """Deterministic (greedy) batched generation. do_sample=False with temperature/top_p/top_k=None
    (never temperature=0.0 together with do_sample=False). Returns only the newly generated text."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    outs = []
    for i in range(0, len(messages_list), batch_size):
        batch = messages_list[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        new_tokens = gen[:, enc["input_ids"].shape[1]:]
        outs += [t.strip() for t in tok.batch_decode(new_tokens, skip_special_tokens=True)]
    return outs

def baseline_messages(query):
    return [{"role": "system", "content": BASELINE_SYSTEM}, {"role": "user", "content": query}]

def rag_messages(query, context):
    return [{"role": "system", "content": RAG_SYSTEM.format(context=context)}, {"role": "user", "content": query}]

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

def get_embeddings():
    # SAME model + settings in every notebook (NB4, NB5, NB7). CPU is enough for MiniLM.
    return HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"},
                                 encode_kwargs={"normalize_embeddings": True})

def load_vector_db(embeddings):
    return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings, collection_name=COLLECTION)

Working directory: /content
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


FileNotFoundError: config.json not found - run Notebook 2 (Data Preparation) first.

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [ ]:
import time
import matplotlib.pyplot as plt, seaborn as sns
from peft import PeftModel

import re, difflib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

SOP_DIR = find_sop_dir()
SOP_RAW = {os.path.basename(f): open(f, encoding="utf-8").read() for f in sorted(glob.glob(os.path.join(SOP_DIR, "*.md")))}
INTENT_TO_SOP = cfg["INTENT_TO_SOP"]
INTENT_SEARCH = cfg["INTENT_SEARCH"]

def sop_sentences(md_text):
    """Split a Markdown SOP into clean sentences (headings dropped, wrapped lines re-joined)."""
    items, cur = [], []
    for line in md_text.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            if cur: items.append(" ".join(cur)); cur = []
            continue
        if re.match(r"^([-*]|\d+[.)])\s+", s):
            if cur: items.append(" ".join(cur))
            cur = [re.sub(r"^([-*]|\d+[.)])\s+", "", s)]
        else:
            cur.append(s)
    if cur: items.append(" ".join(cur))
    sents = []
    for it in items:
        it = re.sub(r"[*_`]", "", it)
        sents += [x.strip() for x in re.split(r"(?<=[.!?])\s+", it) if len(x.split()) >= 4]
    return sents

SOP_SENTS = {k: sop_sentences(v) for k, v in SOP_RAW.items()}

def build_reference(query, sop_file, top_n=4):
    """SOP-grounded reference: the top-N SOP sentences of the CORRECT SOP (known from the true intent)
    that are most similar to the customer query. Identical for every system -> fair comparison."""
    sents = SOP_SENTS[sop_file]
    try:
        vec = TfidfVectorizer(stop_words="english").fit(sents + [query])
        sims = cosine_similarity(vec.transform([query]), vec.transform(sents))[0]
        keep = sorted(np.argsort(-sims)[:top_n])
    except ValueError:
        keep = list(range(min(top_n, len(sents))))
    return " ".join(sents[i] for i in keep)

_rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
_smooth = SmoothingFunction().method1
def words(t): return re.findall(r"[a-z0-9']+", t.lower())

def score_output(output, reference):
    r = _rouge.score(reference, output)
    o, ref = words(output), words(reference)
    bleu = sentence_bleu([ref], o, smoothing_function=_smooth) if o else 0.0
    return r["rouge1"].fmeasure, r["rougeL"].fmeasure, bleu

def format_ok(text):
    """Free-text 'format adherence': non-empty, decodable, at least 5 words."""
    return bool(text) and "\ufffd" not in text and len(text.split()) >= 5

NUM_RE = re.compile(r"\d+(?:\.\d+)?")
ENTITY_RE = re.compile(r"\b(?:[A-Z][A-Za-z]+\s){1,3}(?:Department|Team|Portal|System|Hotline|Center|Centre|Desk|Division|Tracker)\b")
CONTACT_RE = re.compile(r"(https?://\S+|www\.\S+|[\w.+-]+@[\w-]+\.[\w.]+|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b|\b1-800[-\d]*)")
OVERPROMISE_RE = re.compile(r"\b(guarantee[ds]?|definitely|promise[ds]?|within the hour)\b", re.I)

def _strip_list_numbers(t):
    return re.sub(r"(?m)^\s*\d+[.)]\s+", "", t)

def detect_hallucination(output, query, sop_file):
    """Rule-based, reproducible proxy for hallucination. An output is flagged if it contains
       (a) numbers/time-frames that appear neither in the CORRECT SOP nor in the customer query,
       (b) invented departments/systems/portals not named in the SOP,
       (c) contact details (URL / e-mail / phone) not in the SOP,
       (d) over-promising language (guarantee / definitely / promise) - the SOPs tell agents NOT to promise dates."""
    sop = SOP_RAW[sop_file]; sop_l = sop.lower()
    out = _strip_list_numbers(output)
    sop_nums, q_nums = set(NUM_RE.findall(sop)), set(NUM_RE.findall(query))
    bad_nums = sorted({n for n in NUM_RE.findall(out) if n not in sop_nums and n not in q_nums})
    ents = sorted({e for e in ENTITY_RE.findall(out) if e.lower() not in sop_l})
    contacts = sorted({c for c in CONTACT_RE.findall(out) if str(c).lower() not in sop_l})
    over = sorted({m.lower() for m in OVERPROMISE_RE.findall(out)})   # SOPs only use these words in the negative (e.g. 'avoid promising ... a guaranteed date')
    flagged = bool(bad_nums or ents or contacts or over)
    return {"flag": int(flagged), "unsupported_numbers": bad_nums, "invented_entities": ents,
            "invented_contacts": contacts, "over_promises": over}

def pct_change(new, old):
    return float("nan") if (old is None or old == 0 or pd.isna(old) or pd.isna(new)) else (new - old) / old * 100.0

# ---- (1) GENERATOR: the same 4-bit base model + prompts + decoding as Notebook 5 (so only the RETRIEVAL QUERY differs) ----
gen_tok, gen_model = load_base_model(MODEL_ID, quantize=True)

# ---- (2) ROUTER: base model + best LoRA checkpoint, adapters MERGED into the weights for fast inference -------------------
ROUTER_DTYPE = COMPUTE_DTYPE if HAS_CUDA else torch.float32
router_tok = AutoTokenizer.from_pretrained(MODEL_ID); router_tok.padding_side = "left"
if router_tok.pad_token is None: router_tok.pad_token = router_tok.eos_token
router_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=ROUTER_DTYPE, **({"device_map": {"": 0}} if HAS_CUDA else {}))
router_model = PeftModel.from_pretrained(router_base, "./intent_lora_best").merge_and_unload()
router_model.eval()
print("LoRA adapters merged into the router model:", type(router_model).__name__)
# NOTE: the router is a separate copy so the JSON-specialised LoRA weights never touch the free-text synthesis step.

# ---- (3) the SAME vector DB / embedding model as Notebook 4 ------------------------------------------------------------
embeddings = get_embeddings()
vector_db = load_vector_db(embeddings)
RETRIEVAL_K = cfg["RETRIEVAL_K"]
assert vector_db._collection.count() == len(SOP_RAW)
VALID_INTENTS = sorted(INTENT_TO_SOP)

def router_messages(q):          # identical to the training prompt built in Notebook 2
    return [{"role": "system", "content": ROUTER_SYSTEM}, {"role": "user", "content": q}]

def route_batch(queries, max_new_tokens=40, batch_size=32):
    return generate_chat(router_model, router_tok, [router_messages(q) for q in queries], max_new_tokens=max_new_tokens, batch_size=batch_size)

def parse_router(raw):
    """Strict json.loads first; regex safety net for stray preamble. Returns (obj_or_None, strict_valid)."""
    try:
        o = json.loads(raw.strip())
        if isinstance(o, dict): return o, True
    except Exception: pass
    m = re.search(r"\{.*?\}", raw, flags=re.S)
    if m:
        try:
            o = json.loads(m.group(0))
            if isinstance(o, dict): return o, False
        except Exception: pass
    return None, False

def resolve_intent(obj):
    """Map the router output to a valid intent. exact -> in label set; repaired -> nearest label (difflib); None -> fall back to raw query."""
    if not obj or not isinstance(obj.get("intent"), str): return None, None
    it = obj["intent"].strip().lower()
    if it in INTENT_TO_SOP: return it, "exact"
    m = difflib.get_close_matches(it, VALID_INTENTS, n=1, cutoff=0.75)
    return (m[0], "repaired") if m else (None, "unmapped")

def _stem(w): return re.sub(r"(ing|ed|es|s)$", "", w) if len(w) > 4 else w
def intent_tokens(s): return {_stem(w) for w in re.split(r"[_\s]+", s.lower()) if w}
def fuzzy_match(pred, true):
    """Lenient match: identical, same stemmed word-set (order_tracking == track_order) or string similarity >= 0.9."""
    if not pred: return False
    pred = pred.strip().lower()
    return pred == true or intent_tokens(pred) == intent_tokens(true) or difflib.SequenceMatcher(None, pred, true).ratio() >= 0.9
print("Router + generator + vector DB ready.")

In [ ]:
def hybrid_route(queries, raws=None):
    """Router -> JSON intent -> vector-search string -> Chroma retrieval (top-1)."""
    raws = raws if raws is not None else route_batch(queries)
    rows = []
    for q, raw in zip(queries, raws):
        obj, strict = parse_router(raw)
        intent, how = resolve_intent(obj)
        search = INTENT_SEARCH[intent] if intent else q          # fall back to the raw query if parsing/mapping fails
        d = vector_db.similarity_search(search, k=RETRIEVAL_K)[0]
        rows.append({"query": q, "router_raw": raw, "json_strict": strict, "raw_intent": (obj or {}).get("intent"), "raw_category": (obj or {}).get("category"),
                     "pred_intent": intent, "resolve": how, "search_string": search, "hybrid_source": d.metadata["doc_name"], "context": d.page_content})
    return rows

def hybrid_answer(rows):
    """Final synthesis with the SAME generator, system prompt and decoding as Naive RAG."""
    outs = generate_chat(gen_model, gen_tok, [rag_messages(r["query"], r["context"]) for r in rows], batch_size=16)
    return [dict(r, output=o) for r, o in zip(rows, outs)]

def hybrid_pipeline(queries):
    return hybrid_answer(hybrid_route(queries))

# ---- end-to-end validation on the Notebook-3 query: intent -> search string -> retrieved SOP -> answer ---------------------
outputs = json.load(open("outputs.json"))
r = hybrid_pipeline([outputs["test_query"]])[0]
print("Customer query :", r["query"])
print("Router raw     :", r["router_raw"], "| strict JSON:", r["json_strict"])
print("Intent         :", r["pred_intent"], f"({r['resolve']})")
print("Search string  :", r["search_string"])
print("Retrieved SOP  :", r["hybrid_source"])
print("\nHYBRID RAG ANSWER:\n" + "-" * 70 + "\n" + r["output"] + "\n" + "-" * 70)
print("\nNaive RAG answer (Notebook 4) for comparison:\n" + outputs.get("naive_rag_output", "n/a"))
print("\nGround truth:", outputs["ground_truth"], "| mentions 3-7 business days:", bool(re.search(r"\b3\s*(?:-|to|–)\s*7\b", r["output"])))
assert r["hybrid_source"] in SOP_RAW

### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [ ]:
df_test = pd.read_csv("df_test.csv")                      # leakage-free held-out split from Notebook 2
df_test["true_sop"] = df_test["intent"].map(INTENT_TO_SOP)

# ---- (A) route the WHOLE held-out test split -------------------------------------------------------------------------------
t0 = time.time()
rt = pd.DataFrame(hybrid_route(df_test["text"].tolist()))
rt = pd.concat([df_test[["row_id", "intent", "category", "true_sop"]].reset_index(drop=True), rt], axis=1)
print(f"Routed {len(rt)} test queries in {time.time()-t0:.1f}s")
rt["naive_source"] = [vector_db.similarity_search(q, k=RETRIEVAL_K)[0].metadata["doc_name"] for q in rt["query"]]

# ---- (B) adversarial subset: regex filter for sentiment / hedging language -----------------------------------------------------
ADV_REGEX = r"\b(still|never|terrible|frustrat\w*|ridiculous|awful|worst|unacceptable|useless|again|hasn'?t|haven'?t|nothing|wrong|angry|upset|disappointed|not working)\b"
rt["adversarial"] = rt["query"].str.contains(ADV_REGEX, case=False, regex=True)
print(f"Adversarial subset (regex): {int(rt.adversarial.sum())} of {len(rt)} test rows")
if rt.adversarial.sum() < 20:
    print("  note: small adversarial subset - read its percentages with caution (wide confidence interval).")

def router_metrics(sub):
    if len(sub) == 0: return {k: np.nan for k in ["n", "Format adherence - strict JSON (%)", "Format adherence - parseable after regex (%)", "Intent exact match (%)", "Category exact match (%)", "Intent fuzzy match (%)", "Hybrid retrieval accuracy (%)", "Naive retrieval accuracy (%)"]}
    return {"n": len(sub),
            "Format adherence - strict JSON (%)": sub.json_strict.mean() * 100,
            "Format adherence - parseable after regex (%)": sub.raw_intent.notna().mean() * 100,
            "Intent exact match (%)": (sub.raw_intent.astype(str).str.strip().str.lower() == sub.intent).mean() * 100,
            "Category exact match (%)": (sub.raw_category.astype(str).str.strip().str.upper() == sub.category).mean() * 100,
            "Intent fuzzy match (%)": np.mean([fuzzy_match(p if isinstance(p, str) else None, t) for p, t in zip(sub.raw_intent, sub.intent)]) * 100,
            "Hybrid retrieval accuracy (%)": (sub.hybrid_source == sub.true_sop).mean() * 100,
            "Naive retrieval accuracy (%)": (sub.naive_source == sub.true_sop).mean() * 100}

router_tbl = pd.DataFrame({"Full held-out test": router_metrics(rt), "Adversarial subset": router_metrics(rt[rt.adversarial])}).round(1)
display(router_tbl)
v2_router = router_tbl.to_dict()

print("\nMost frequent router mistakes (true -> predicted):")
wrong = rt[rt.raw_intent.astype(str).str.strip().str.lower() != rt.intent]
print(wrong.groupby(["intent", "raw_intent"]).size().sort_values(ascending=False).head(8).to_string() if len(wrong) else "none")
print("\nExamples of router outputs that were not strict JSON:", rt.loc[~rt.json_strict, "router_raw"].head(3).tolist() or "none")

# ---- (C) final synthesis on the same rows as Notebook 5 (SOP-grounded ROUGE / BLEU, hallucination, consistency) -------------------
v1 = pd.read_csv("v1_metrics.csv")
v1_extra = json.load(open("v1_extra.json"))
sub = rt[rt.row_id.isin(v1.row_id)].set_index("row_id").loc[v1.row_id].reset_index()          # same rows + same order as V1
t1 = time.time()
hyb = pd.DataFrame(hybrid_answer(sub.to_dict("records")))
print(f"Hybrid synthesis for {len(hyb)} rows in {(time.time()-t1)/60:.1f} min")

hyb["reference"] = v1["reference"].values                                  # identical SOP-grounded reference for all systems
sc = [score_output(o, ref) for o, ref in zip(hyb["output"], hyb["reference"])]
hyb["rouge1_hyb"], hyb["rougeL_hyb"], hyb["bleu_hyb"] = zip(*sc)
hyb["wellformed_hyb"] = hyb["output"].map(format_ok).astype(int)
hyb["halluc_hyb"] = [detect_hallucination(o, q, s)["flag"] for o, q, s in zip(hyb["output"], hyb["query"], hyb["true_sop"])]
hyb["hybrid_retrieval_correct"] = (hyb["hybrid_source"] == hyb["true_sop"]).astype(int)

# consistency: same 30 queries as Notebook 5, 3 full pipeline runs (router + retrieval + generation)
N_CONSIST = v1_extra.get("n_eval", len(v1)); N_CONSIST = min(30, N_CONSIST)
cq = v1.sample(N_CONSIST, random_state=SEED)["query"].tolist()
runs = [[x["output"] for x in hybrid_pipeline(cq)] for _ in range(3)]
cons_hyb = np.mean([len({runs[k][i] for k in range(3)}) == 1 for i in range(N_CONSIST)]) * 100
print(f"Consistency (Hybrid, {N_CONSIST} queries x 3 runs): {cons_hyb:.1f}%")

print(f"\nHybrid synthesis -> ROUGE-1 {hyb.rouge1_hyb.mean():.4f} | ROUGE-L {hyb.rougeL_hyb.mean():.4f} | BLEU {hyb.bleu_hyb.mean():.4f} | hallucination {hyb.halluc_hyb.mean()*100:.1f}%")

#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [ ]:
v1_sum = pd.read_csv("v1_summary.csv").set_index("Metric")
rows = [
    ("Format adherence - well-formed (%)", v1.wellformed_naive.mean()*100, hyb.wellformed_hyb.mean()*100, True),
    ("Retrieval accuracy (%)",             v1.naive_retrieval_correct.mean()*100, hyb.hybrid_retrieval_correct.mean()*100, True),
    ("ROUGE-1",                            v1.rouge1_naive.mean(), hyb.rouge1_hyb.mean(), True),
    ("ROUGE-L",                            v1.rougeL_naive.mean(), hyb.rougeL_hyb.mean(), True),
    ("BLEU",                               v1.bleu_naive.mean(),   hyb.bleu_hyb.mean(), True),
    ("Consistency rate (%)",               v1_extra["consistency_naive"], cons_hyb, True),
    ("Hallucination frequency (%)",        v1.halluc_naive.mean()*100, hyb.halluc_hyb.mean()*100, False),
]
ft = pd.DataFrame(rows, columns=["Metric", "Solution V1 (Naive RAG)", "Solution V2 (Hybrid RAG)", "higher_is_better"])
ft["Change (abs)"] = ft["Solution V2 (Hybrid RAG)"] - ft["Solution V1 (Naive RAG)"]
ft["Change (%)"] = [pct_change(b, a) for a, b in zip(ft["Solution V1 (Naive RAG)"], ft["Solution V2 (Hybrid RAG)"])]
ft["Improvement (%)"] = [c if h else -c for c, h in zip(ft["Change (%)"], ft["higher_is_better"])]
display(ft.drop(columns="higher_is_better").round(4))

# paired bootstrap 95% CI for the per-row differences (same rows for V1 and V2)
rng = np.random.default_rng(SEED)
def boot_ci(diff, n=2000):
    m = [rng.choice(diff, len(diff), replace=True).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])
d_rl = hyb.rougeL_hyb.values - v1.rougeL_naive.values
d_h = (hyb.halluc_hyb.values - v1.halluc_naive.values) * 100
d_r = (hyb.hybrid_retrieval_correct.values - v1.naive_retrieval_correct.values) * 100
for name, d in [("ROUGE-L", d_rl), ("Hallucination (pp)", d_h), ("Retrieval accuracy (pp)", d_r)]:
    lo, hi = boot_ci(d)
    print(f"V2 - V1 {name:25}: mean {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  -> {'significant' if lo > 0 or hi < 0 else 'not significant'}")

ra = ft.set_index("Metric")
print(f"\nATTRIBUTION: V1 and V2 share the same base model, SOP corpus, embedding model, k=1, prompts and decoding. "
      f"The only change is the retrieval query (raw text vs router-derived intent string), so the gain "
      f"(retrieval accuracy {ra.loc['Retrieval accuracy (%)','Change (abs)']:+.1f} pp, ROUGE-L {ra.loc['ROUGE-L','Change (%)']:+.1f}%, "
      f"hallucination {ra.loc['Hallucination frequency (%)','Change (abs)']:+.1f} pp) is attributable to the fine-tuned router.")
resid = hyb[hyb.hybrid_retrieval_correct == 0]
print(f"Residual errors: {len(resid)} of {len(hyb)} rows still retrieve a different SOP than the intent->SOP map; router intent errors are the main cause "
      f"(see the confusion list above).")

### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [ ]:
# ---- per-row results for ALL THREE architectures (identical test rows, references and decoding settings) -------------------
full = v1[["row_id", "query", "intent", "category", "true_sop", "reference", "baseline_output", "naive_output", "naive_source",
           "naive_retrieval_correct", "wellformed_base", "rouge1_base", "rougeL_base", "bleu_base", "halluc_base",
           "wellformed_naive", "rouge1_naive", "rougeL_naive", "bleu_naive", "halluc_naive"]].copy()
full = full.merge(hyb[["row_id", "router_raw", "json_strict", "raw_intent", "pred_intent", "search_string", "hybrid_source", "output", "hybrid_retrieval_correct",
                       "wellformed_hyb", "rouge1_hyb", "rougeL_hyb", "bleu_hyb", "halluc_hyb"]].rename(columns={"output": "hybrid_output"}), on="row_id", how="left")
full.to_csv("Comparative_Results_Full.csv", index=False)

S = pd.DataFrame({
    "Metric": ["Format adherence - well-formed (%)", "Router JSON adherence - strict (%)", "Intent accuracy - exact (%)", "Intent accuracy - fuzzy (%)",
               "Retrieval accuracy (%)", "ROUGE-1", "ROUGE-L", "BLEU", "Consistency rate (%)", "Hallucination frequency (%)"],
    "Baseline": [full.wellformed_base.mean()*100, np.nan, np.nan, np.nan, np.nan, full.rouge1_base.mean(), full.rougeL_base.mean(), full.bleu_base.mean(), v1_extra["consistency_baseline"], full.halluc_base.mean()*100],
    "Naive RAG (V1)": [full.wellformed_naive.mean()*100, np.nan, np.nan, np.nan, full.naive_retrieval_correct.mean()*100, full.rouge1_naive.mean(), full.rougeL_naive.mean(), full.bleu_naive.mean(), v1_extra["consistency_naive"], full.halluc_naive.mean()*100],
    "Hybrid RAG (V2)": [full.wellformed_hyb.mean()*100, router_tbl.loc["Format adherence - strict JSON (%)", "Full held-out test"], router_tbl.loc["Intent exact match (%)", "Full held-out test"],
                        router_tbl.loc["Intent fuzzy match (%)", "Full held-out test"], full.hybrid_retrieval_correct.mean()*100, full.rouge1_hyb.mean(), full.rougeL_hyb.mean(), full.bleu_hyb.mean(), cons_hyb, full.halluc_hyb.mean()*100],
})
S["V1 vs Baseline (%)"] = [pct_change(b, a) for a, b in zip(S["Baseline"], S["Naive RAG (V1)"])]
S["V2 vs V1 (%)"] = [pct_change(b, a) for a, b in zip(S["Naive RAG (V1)"], S["Hybrid RAG (V2)"])]
S["V2 vs Baseline (%)"] = [pct_change(b, a) for a, b in zip(S["Baseline"], S["Hybrid RAG (V2)"])]
S.to_csv("Comparative_Results_Summary.csv", index=False)
display(S.round(4))

# ---- charts ----------------------------------------------------------------------------------------------------------------------
cols = ["#b0b7c3", "#3b6fb6", "#2a9d8f"]
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
for a, m in zip(ax, ["ROUGE-L", "BLEU", "Hallucination frequency (%)", "Retrieval accuracy (%)"]):
    vals = S.set_index("Metric").loc[m, ["Baseline", "Naive RAG (V1)", "Hybrid RAG (V2)"]].astype(float)
    a.bar(["Baseline", "Naive\nRAG (V1)", "Hybrid\nRAG (V2)"], vals.fillna(0).values, color=cols); a.set_title(m)
    for i, v in enumerate(vals.values):
        a.text(i, (0 if np.isnan(v) else v), "n/a" if np.isnan(v) else f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig("comparison_chart.png", dpi=150); plt.show()
print("Saved Comparative_Results_Full.csv", full.shape, "and Comparative_Results_Summary.csv", S.shape)

In [ ]:
# ===== Generates Comparative_Analysis_Report.pdf from the REAL numbers computed above (nothing is hard-coded) =====
import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

M = S.set_index("Metric")
g = lambda m, c: M.loc[m, c]
def fmt(v, m): return "n/a" if pd.isna(v) else (f"{v:.1f}" if "%" in m else f"{v:.4f}")
def chg(v): return "n/a" if pd.isna(v) else f"{v:+.1f}%"
better = lambda a, b, hib=True: (b > a) if hib else (b < a)

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=ss["Heading1"], fontSize=15, spaceBefore=12, spaceAfter=6, textColor=colors.HexColor("#1f3a5f"))
BODY = ParagraphStyle("B", parent=ss["BodyText"], fontSize=9.5, leading=13.5)
BUL = ParagraphStyle("BL", parent=BODY, leftIndent=14, bulletIndent=4)
SMALL = ParagraphStyle("S", parent=BODY, fontSize=8, textColor=colors.grey)

def bullets(items): return [Paragraph(t, BUL, bulletText="•") for t in items]

def table(data, widths=None):
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f3a5f")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                           ("FONTSIZE", (0, 0), (-1, -1), 8), ("GRID", (0, 0), (-1, -1), 0.4, colors.lightgrey),
                           ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f2f5f9")]), ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    return t

n_eval = len(full)
# ---- results table
res = [["Metric", "Baseline", "Naive RAG (V1)", "Hybrid RAG (V2)", "V1 vs Base", "V2 vs V1", "V2 vs Base"]]
for m in S["Metric"]:
    res.append([m] + [fmt(g(m, c), m) for c in ["Baseline", "Naive RAG (V1)", "Hybrid RAG (V2)"]] + [chg(g(m, c)) for c in ["V1 vs Baseline (%)", "V2 vs V1 (%)", "V2 vs Baseline (%)"]])

# ---- findings (computed from the numbers)
rl_b, rl_1, rl_2 = g("ROUGE-L", "Baseline"), g("ROUGE-L", "Naive RAG (V1)"), g("ROUGE-L", "Hybrid RAG (V2)")
h_b, h_1, h_2 = g("Hallucination frequency (%)", "Baseline"), g("Hallucination frequency (%)", "Naive RAG (V1)"), g("Hallucination frequency (%)", "Hybrid RAG (V2)")
ra_1, ra_2 = g("Retrieval accuracy (%)", "Naive RAG (V1)"), g("Retrieval accuracy (%)", "Hybrid RAG (V2)")
ia, fz, fa = g("Intent accuracy - exact (%)", "Hybrid RAG (V2)"), g("Intent accuracy - fuzzy (%)", "Hybrid RAG (V2)"), g("Router JSON adherence - strict (%)", "Hybrid RAG (V2)")
adv = router_tbl["Adversarial subset"]
adv_n = 0 if pd.isna(adv['n']) else int(adv['n'])
best_rl = ["Baseline", "Naive RAG (V1)", "Hybrid RAG (V2)"][int(np.nanargmax([rl_b, rl_1, rl_2]))]
best_h = ["Baseline", "Naive RAG (V1)", "Hybrid RAG (V2)"][int(np.nanargmin([h_b, h_1, h_2]))]
findings = [
    f"<b>Retrieval impact (Baseline to V1).</b> Grounding the LLM in an SOP changed ROUGE-L from {rl_b:.4f} to {rl_1:.4f} ({chg(g('ROUGE-L','V1 vs Baseline (%)'))}) and hallucination frequency from {h_b:.1f}% to {h_1:.1f}%. "
    f"Naive retrieval selected the correct SOP for only {ra_1:.1f}% of raw customer queries, which caps what V1 can achieve.",
    f"<b>Fine-tuning impact (V1 to V2).</b> The LoRA router produced strictly valid JSON for {fa:.1f}% of test queries, with {ia:.1f}% exact and {fz:.1f}% fuzzy intent accuracy. "
    f"Retrieval accuracy moved from {ra_1:.1f}% to {ra_2:.1f}%, ROUGE-L from {rl_1:.4f} to {rl_2:.4f} ({chg(g('ROUGE-L','V2 vs V1 (%)'))}) and hallucination frequency from {h_1:.1f}% to {h_2:.1f}%.",
    f"<b>Adversarial (sentiment/hedging) subset, n={adv_n}.</b> Intent exact match {adv['Intent exact match (%)']:.1f}%, hybrid retrieval accuracy {adv['Hybrid retrieval accuracy (%)']:.1f}% "
    f"versus naive retrieval accuracy {adv['Naive retrieval accuracy (%)']:.1f}%.",
    f"<b>Best system.</b> Highest ROUGE-L: {best_rl}. Lowest hallucination frequency: {best_h}. Consistency under greedy decoding: "
    f"{g('Consistency rate (%)','Baseline'):.0f}% / {g('Consistency rate (%)','Naive RAG (V1)'):.0f}% / {g('Consistency rate (%)','Hybrid RAG (V2)'):.0f}% (Baseline / V1 / V2).",
]
hybrid_wins = better(ra_1, ra_2) and better(h_1, h_2, hib=False)
recs = [
    ("Deploy the Hybrid RAG pipeline (router + retrieval) as the default path." if hybrid_wins else
     "The Hybrid pipeline did not dominate on every metric in this run; review the per-metric table before deciding, and consider more training steps or more data for the router."),
    "Keep the fallback to the raw query when the router output cannot be parsed or mapped; log these cases for monitoring.",
    "Route low-confidence or unmapped intents to a human agent (escalation_matrix SOP) rather than answering.",
    "Re-train the adapter whenever intents or SOPs change; keep the intent-to-SOP map under version control next to the SOPs.",
]
limits = [
    "Hallucination frequency is a rule-based proxy (unsupported numbers, invented departments, contact details, over-promising). It cannot see purely semantic errors; a human or LLM-judge audit is recommended.",
    "ROUGE/BLEU reward lexical overlap with SOP-grounded references (top-4 query-relevant sentences of the correct SOP). They do not measure helpfulness or tone.",
    "The ground-truth intent-to-SOP mapping is a documented design assumption (several Bitext intents have no dedicated SOP and share the closest one), so retrieval accuracy depends on it.",
    "Single training run and a single seed; a 1.5B 4-bit model. Differences of a few points may be within noise (see bootstrap intervals in the notebook).",
    "Bitext queries are synthetic and mostly clean; real customer messages may be noisier than the adversarial subset used here.",
    "Whole-document indexing with a 256-token embedding window: the tail of long SOPs is not embedded (the full text is still given to the LLM).",
]
future = [
    "Add a reranker or hybrid lexical+dense retrieval and evaluate k=3 with a cross-encoder.",
    "Chunk SOPs by section and let the router emit a topic as well as an intent for finer-grained retrieval.",
    "Train for a full epoch with a larger sample and compare r=8/16/32 and extra target modules (k_proj, o_proj).",
    "Add an LLM-as-judge or human faithfulness review and multi-turn evaluation.",
]

doc = SimpleDocTemplate("Comparative_Analysis_Report.pdf", pagesize=A4, leftMargin=1.8*cm, rightMargin=1.8*cm, topMargin=1.6*cm, bottomMargin=1.6*cm,
                        title="Comparative Analysis Report", author="Hybrid RAG & Fine-Tuning Capstone")
st = [Paragraph("Comparative Analysis Report", ParagraphStyle("T", parent=ss["Title"], fontSize=22, textColor=colors.HexColor("#1f3a5f"))),
      Paragraph("Hybrid RAG &amp; Fine-Tuning for Customer Support: Baseline vs Naive RAG vs Hybrid RAG", ss["Heading3"]),
      Paragraph(f"Generated {datetime.date.today():%d %B %Y} from Notebook 7 | model {MODEL_ID} | embeddings {EMBED_MODEL} | {n_eval} held-out test rows", SMALL),
      Paragraph("1. Objective and setup", H1),
      Paragraph("This report compares three progressively enhanced systems on the same held-out test split, using identical SOP-grounded references and identical deterministic decoding "
                "(do_sample=False, temperature=None): <b>(1) Baseline</b> - the pre-trained model with no context; <b>(2) Naive RAG</b> - the raw customer query retrieves the top-1 SOP from Chroma; "
                "<b>(3) Hybrid RAG</b> - a QLoRA fine-tuned router extracts a JSON intent, the intent is mapped to a vector-search string, the matching SOP is retrieved and the base model writes the answer.", BODY),
      Spacer(1, 6),
      table([["Setting", "Value"], ["Base model", MODEL_ID + " (4-bit NF4)"], ["Embedding / vector DB", EMBED_MODEL + " / ChromaDB (cosine, top-1)"],
             ["Fine-tuning", "LoRA r=16, alpha=32, q_proj+v_proj; AdamW; best checkpoint by validation loss"], ["Evaluation rows", f"{n_eval} (held-out test split, zero leakage)"],
             ["Adversarial subset", f"{adv_n} regex-filtered rows (still / never / terrible / frustrated ...)"]], [4.5*cm, 12.5*cm]),
      Paragraph("2. Results (all metrics, all versions)", H1), table(res, [5.4*cm, 1.6*cm, 2.1*cm, 2.2*cm, 1.8*cm, 1.8*cm, 1.9*cm]),
      Paragraph("Higher is better except hallucination frequency. n/a = metric not applicable to that system. Percent columns are relative changes.", SMALL), Spacer(1, 6),
      Image("comparison_chart.png", width=17*cm, height=3.8*cm),
      Paragraph("3. Key findings", H1)] + bullets(findings)
if os.path.exists("training_curves.png"):
    st += [Paragraph("4. Fine-tuning evidence", H1), Image("training_curves.png", width=16*cm, height=4.8*cm),
           Paragraph("Train vs validation loss; the checkpoint with the lowest validation loss is the one used by the router.", SMALL)]
st += [Paragraph("5. Limitations", H1)] + bullets(limits) + [Paragraph("6. Recommendations", H1)] + bullets(recs) + [Paragraph("7. Future work", H1)] + bullets(future)
doc.build(st)
print("Saved Comparative_Analysis_Report.pdf")

---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**